# Lesson 43: Instance Segmentation

Semantic segmentation (Lesson 42) labels each pixel with a class, but it has no notion of *how many* instances of each class there are — two touching instances are just labeled as one connected blob of pixels. **Instance segmentation** answers the harder question: which pixels belong to *this specific* object, as opposed to some other object of the same class. This lesson augments Lesson 42's segmentation network directly to solve it, with an idea that traces straight back to Lesson 12's Hough transform: instead of only classifying pixels, have the network also predict, for every foreground pixel, a vote for where its object's center is — then cluster the votes to recover individual instances.

In [ ]:
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Overlapping instances

Two circles per scene, deliberately placed close enough to touch or overlap. Ground truth includes both a semantic mask (0=background, 1=circle) and an *instance* mask (0=background, 1=first circle, 2=second circle).

In [ ]:
SIZE = 32

def make_scene(rng, size=SIZE, n_circles=2, r=5):
    scene = np.zeros((size, size), dtype=np.float32)
    sem_mask = np.zeros((size, size), dtype=np.int64)
    inst_mask = np.zeros((size, size), dtype=np.int64)
    centers = []
    for k in range(n_circles):
        if k == 0:
            cx, cy = rng.integers(r + 8, size - r - 8), rng.integers(r + 8, size - r - 8)
        else:
            px, py = centers[-1]
            angle = rng.uniform(0, 2 * np.pi)
            dist = rng.uniform(6, 9)  # overlapping (< 2r) but centers still separable
            cx = int(np.clip(px + dist * np.cos(angle), r, size - r - 1))
            cy = int(np.clip(py + dist * np.sin(angle), r, size - r - 1))
        centers.append((cx, cy))
        yy, xx = np.mgrid[0:size, 0:size]
        m = ((xx - cx) ** 2 + (yy - cy) ** 2) <= r ** 2
        scene[m] = 1.0
        sem_mask[m] = 1
        inst_mask[m] = k + 1  # later circles paint over earlier ones at overlaps
    scene = np.clip(scene + rng.normal(0, 0.05, scene.shape), 0, 1).astype(np.float32)
    return scene, sem_mask, inst_mask, centers

rng = np.random.default_rng(17)
N = 300
scenes, sem_masks, inst_masks, all_centers = [], [], [], []
for _ in range(N):
    s, sm, im, c = make_scene(rng)
    scenes.append(s); sem_masks.append(sm); inst_masks.append(im); all_centers.append(c)
scenes = np.array(scenes, dtype=np.float32)
sem_masks = np.array(sem_masks, dtype=np.int64)

fig, axes = plt.subplots(2, 4, figsize=(9, 4.5))
for i in range(4):
    axes[0, i].imshow(scenes[i], cmap='gray'); axes[0, i].axis('off')
    axes[1, i].imshow(inst_masks[i], cmap='viridis'); axes[1, i].axis('off')
axes[0, 0].set_title('input', fontsize=9, loc='left')
axes[1, 0].set_title('ground truth instance mask', fontsize=9, loc='left')
plt.show()

## Augmenting the network with an instance head

Take Lesson 42's exact U-Net architecture and give it a second output head alongside the semantic head: for every foreground pixel, predict a `(dx, dy)` offset pointing toward its own instance's center. The semantic head still does the same job as Lesson 42 (background vs. circle); the new offset head is what turns plain segmentation into *instance* segmentation.

In [ ]:
# build offset targets: for every foreground pixel, the (dx, dy) to ITS instance's center
yy, xx = np.mgrid[0:SIZE, 0:SIZE]
offset_targets = np.zeros((N, 2, SIZE, SIZE), dtype=np.float32)
for i in range(N):
    im = inst_masks[i]
    for k, (cx, cy) in enumerate(all_centers[i]):
        m = im == (k + 1)
        offset_targets[i, 0][m] = (cx - xx[m]) / SIZE
        offset_targets[i, 1][m] = (cy - yy[m]) / SIZE

split = int(0.85 * N)
Xtr, Str, Otr = scenes[:split], sem_masks[:split], offset_targets[:split]
Xte, Ste, Ote = scenes[split:], sem_masks[split:], offset_targets[split:]
inst_te, centers_te = inst_masks[split:], all_centers[split:]

class InstanceNet(nn.Module):
    """Lesson 42's U-Net, with a second head: per-pixel offset-to-center regression"""
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU())
        self.enc2 = nn.Sequential(nn.Conv2d(16, 32, 3, padding=1), nn.ReLU())
        self.pool = nn.MaxPool2d(2)
        self.up = nn.Upsample(scale_factor=2, mode='nearest')
        self.dec1 = nn.Sequential(nn.Conv2d(32 + 16, 16, 3, padding=1), nn.ReLU())
        self.sem_head = nn.Conv2d(16, 2, 1)     # background vs. circle
        self.offset_head = nn.Conv2d(16, 2, 1)  # (dx, dy) to this pixel's instance center

    def forward(self, x):
        f1 = self.enc1(x)
        f2 = self.enc2(self.pool(f1))
        d1 = self.dec1(torch.cat([self.up(f2), f1], dim=1))
        return self.sem_head(d1), self.offset_head(d1)

torch.manual_seed(0)
model = InstanceNet()
opt = torch.optim.Adam(model.parameters(), lr=0.01)
Xt = torch.tensor(Xtr).unsqueeze(1); St = torch.tensor(Str); Ot = torch.tensor(Otr)
fg_mask_t = (St == 1).unsqueeze(1).float()

for _ in range(300):
    opt.zero_grad()
    sem_logits, offset_pred = model(Xt)
    sem_loss = F.cross_entropy(sem_logits, St)
    offset_loss = (F.mse_loss(offset_pred, Ot, reduction='none') * fg_mask_t).sum() / fg_mask_t.sum().clamp(min=1)
    (sem_loss + 2.0 * offset_loss).backward()
    opt.step()

with torch.no_grad():
    sem_logits_te, offset_pred_te = model(torch.tensor(Xte).unsqueeze(1))
    sem_preds = sem_logits_te.argmax(1).numpy()
    offset_preds = offset_pred_te.numpy()

print(f'semantic pixel accuracy: {(sem_preds == Ste).mean():.1%}')


## Recovering instances: vote for the center, then cluster (Hough, again)

The offset head above was trained to predict, for every foreground pixel, an offset pointing toward *its own instance's* center — supervised directly from the ground-truth instance masks. Each foreground pixel casts one vote (its predicted center location) into an accumulator, exactly like Lesson 12's Hough line/circle voting. Because every pixel belonging to the same circle votes for approximately the same point, the accumulator forms one tight cluster of votes per instance, however tangled the pixels themselves are. Finding instances becomes: find the vote clusters, then assign each pixel to its nearest cluster.

In [ ]:
def cluster_instances(binary_mask, offset_pred, size=SIZE, peak_dist=4, vote_thresh=3):
    ys, xs = np.where(binary_mask)
    if len(xs) == 0:
        return np.zeros_like(binary_mask, dtype=np.int64), []
    pred_cx = xs + offset_pred[0][ys, xs] * size
    pred_cy = ys + offset_pred[1][ys, xs] * size

    votes = np.zeros((size, size), dtype=np.float32)
    for cx, cy in zip(pred_cx, pred_cy):
        cxi = int(np.clip(round(cx), 0, size - 1)); cyi = int(np.clip(round(cy), 0, size - 1))
        votes[cyi, cxi] += 1

    # greedily take the highest-voted peak, suppress its neighborhood, repeat
    # (the same non-max-suppression idea as Lesson 40's detection boxes)
    peaks = []
    votes_work = votes.copy()
    for _ in range(6):
        idx = np.unravel_index(votes_work.argmax(), votes_work.shape)
        if votes_work[idx] < vote_thresh:
            break
        peaks.append((idx[1], idx[0]))
        y0, y1 = max(0, idx[0] - peak_dist), min(size, idx[0] + peak_dist + 1)
        x0, x1 = max(0, idx[1] - peak_dist), min(size, idx[1] + peak_dist + 1)
        votes_work[y0:y1, x0:x1] = 0

    if not peaks:
        return np.zeros_like(binary_mask, dtype=np.int64), []
    inst_pred = np.zeros_like(binary_mask, dtype=np.int64)
    peaks_arr = np.array(peaks)
    for y, x in zip(ys, xs):
        dists = (peaks_arr[:, 0] - x) ** 2 + (peaks_arr[:, 1] - y) ** 2
        inst_pred[y, x] = np.argmin(dists) + 1
    return inst_pred, peaks

n_correct = 0
inst_preds, all_peaks = [], []
for i in range(len(Xte)):
    binary_mask = sem_preds[i] == 1
    inst_pred, peaks = cluster_instances(binary_mask, offset_preds[i])
    inst_preds.append(inst_pred); all_peaks.append(peaks)
    true_n = len(set(inst_te[i].ravel()) - {0})
    if len(peaks) == true_n:
        n_correct += 1

print(f'offset-voting recovers the correct instance count: {n_correct}/{len(Xte)} scenes '
      f'({n_correct/len(Xte):.1%})')

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(9, 6.5))
for i in range(4):
    axes[0, i].imshow(scenes[split + i], cmap='gray'); axes[0, i].axis('off')
    axes[1, i].imshow(inst_te[i], cmap='viridis'); axes[1, i].axis('off')
    axes[2, i].imshow(inst_preds[i], cmap='viridis'); axes[2, i].axis('off')
for r, name in enumerate(['input', 'ground truth instances', 'recovered via offset voting']):
    axes[r, 0].set_title(name, fontsize=9, loc='left')
plt.tight_layout()
plt.show()

## Mask R-CNN: Faster R-CNN plus a mask branch

The offset-voting approach above is one family of instance segmentation methods (related to techniques sometimes called "instance embedding" or center-voting). The other dominant approach, **Mask R-CNN** (<a href="../references.html#he-2017-maskrcnn">He et al., 2017</a><span class="landmark-paper">&#9733;</span>), does not start from scratch — it is **Faster R-CNN** (Lesson 41) with one new piece bolted on. The region proposal network, the per-box classifier, and the per-box coordinate refinement are all unchanged. Mask R-CNN just adds a third, parallel branch on every detected box: alongside the existing class label and refined box coordinates, a small fully-convolutional head predicts a binary mask *within that box*. Detection already solves "how many objects, and roughly where" via region proposals and NMS; the mask branch adds "and here’s this one’s exact silhouette" — exactly the piece built and tested in isolation below.

One more change was needed to make that mask branch actually work well. Faster R-CNN extracts a fixed-size feature grid for each proposed box by snapping (quantizing) the box’s coordinates to the nearest feature-map cell — a step called RoIPool. That rounding barely affects classifying a box or nudging its coordinates by a few pixels, but it visibly misaligns a per-pixel mask against the object it is supposed to outline. Mask R-CNN replaces RoIPool with **RoIAlign**, which reads features via bilinear interpolation instead of rounding to the nearest cell, so the extracted features — and therefore the predicted mask — stay aligned with the actual box coordinates. This is a small change to the detector, but it is the specific fix that makes pixel-accurate masks possible.

Putting Lessons 36, 40, 41, and this lesson together: **classification** (Lesson 36) gives one label for the whole image, **semantic segmentation** (Lesson 42) gives one label per pixel with no notion of separate objects of the same class, and **instance segmentation** (this lesson) adds a distinct identity per object — "background," "circle #1," "circle #2," not just "background," "circle." A fourth term, **panoptic segmentation**, unifies the last two: every pixel gets both a semantic class and, for countable "thing" classes like circles or cars, an instance ID — while uncountable "stuff" classes like sky or road are labeled semantically only, since instance identity doesn’t make sense for them.

## A mask head

Detection (Lesson 41) and per-instance masking are two separate, already-solved pieces at this point — the only genuinely new ingredient Mask R-CNN adds is the **mask head**: a small network that looks *only* inside one proposed box and predicts a binary mask for the single object that box was proposed for, ignoring anything else visible in the crop.

Build that piece in isolation. Take the ground-truth box around each instance's true center (assume localization is already solved, so only the new mask-head idea is under test), crop a fixed 14x14 window there, and train a tiny fully-convolutional head to predict which pixels in that crop belong to *this* instance — not to any neighboring circle that happens to also be visible in the same crop.

In [ ]:
CROP = 14
HALF = CROP // 2

def make_crops(indices):
    crops, masks = [], []
    for i in indices:
        scene, inst = scenes[i], inst_masks[i]
        for k, (cx, cy) in enumerate(all_centers[i]):
            x0 = int(np.clip(cx - HALF, 0, SIZE - CROP))
            y0 = int(np.clip(cy - HALF, 0, SIZE - CROP))
            crop = scene[y0:y0 + CROP, x0:x0 + CROP]
            inst_crop = inst[y0:y0 + CROP, x0:x0 + CROP]
            mask = (inst_crop == (k + 1)).astype(np.float32)  # only THIS instance, never the neighbor
            crops.append(crop); masks.append(mask)
    return np.array(crops, dtype=np.float32), np.array(masks, dtype=np.float32)

X_crop_train, M_crop_train = make_crops(range(split))
X_crop_test, M_crop_test = make_crops(range(split, N))
print(f'{len(X_crop_train)} training crops, {len(X_crop_test)} test crops, each {CROP}x{CROP}')

class MaskHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 1, 1),  # per-pixel mask logit, full crop resolution throughout
        )

    def forward(self, x):
        return self.net(x)

torch.manual_seed(0)
mask_head = MaskHead()
opt = torch.optim.Adam(mask_head.parameters(), lr=0.01)
Xc = torch.tensor(X_crop_train).unsqueeze(1)
Mc = torch.tensor(M_crop_train).unsqueeze(1)
for _ in range(300):
    opt.zero_grad()
    loss = F.binary_cross_entropy_with_logits(mask_head(Xc), Mc)
    loss.backward()
    opt.step()

with torch.no_grad():
    pred_masks = (mask_head(torch.tensor(X_crop_test).unsqueeze(1)) > 0).float().squeeze(1).numpy()

def mean_iou(pred, true):
    inter = (pred * true).sum(axis=(1, 2))
    union = ((pred + true) > 0).sum(axis=(1, 2))
    return (inter / np.clip(union, 1, None)).mean()

naive_masks = (X_crop_test > 0.5).astype(np.float32)  # "everything foreground in the crop", no per-instance awareness
print(f'mask-head IoU (per-instance):          {mean_iou(pred_masks, M_crop_test):.3f}')
print(f'naive foreground-in-crop baseline IoU: {mean_iou(naive_masks, M_crop_test):.3f}')

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(9, 5))
for i in range(4):
    axes[0, i].imshow(X_crop_test[i], cmap='gray'); axes[0, i].axis('off')
    axes[1, i].imshow(M_crop_test[i], cmap='viridis', vmin=0, vmax=1); axes[1, i].axis('off')
    axes[2, i].imshow(pred_masks[i], cmap='viridis', vmin=0, vmax=1); axes[2, i].axis('off')
for r, name in enumerate(['crop (neighbor circle visible)', 'ground truth mask (this instance only)', 'predicted mask']):
    axes[r, 0].set_title(name, fontsize=9, loc='left')
plt.tight_layout()
plt.show()

Because this dataset's circles are generated close enough to always touch or overlap, *every single test crop* contains visible pixels from the neighboring circle — there is no "easy" case here to inflate the numbers. The per-instance mask head still reaches a respectably high IoU against the true single-instance mask, clearly ahead of the naive baseline that just marks every foreground pixel in the crop as belonging to "the" object (which is systematically wrong whenever a neighbor intrudes — the same problem that motivated giving the network an instance-center head in the first place, just replayed at crop scale). The mask head doesn't need to solve "where are the circles" — the box already answered that — it only has to learn "given that this box was proposed for one specific circle, which pixels are that circle's, and not its neighbor's." That's a strictly easier, more local question, which is exactly why Mask R-CNN's mask branch can be small and fast: a good box already does most of the work.

This also clarifies why Mask R-CNN and this lesson's offset-voting approach rarely fail the same way. Offset-voting can mis-cluster votes when two instances' centers are close enough that their accumulator peaks blur together (Exercise 1 below explores this). Mask R-CNN instead depends on the upstream detector proposing a correct box for every instance in the first place — miss a box, and that instance's mask is never even attempted, no matter how good the mask head is.

## In practice: Mask R-CNN on a real photo

The mask head above proves the idea works once localization is solved for free (ground-truth box centers). Here is the same idea, fully assembled and pretrained: a real **Mask R-CNN** (`torchvision.models.detection.maskrcnn_resnet50_fpn`, trained on COCO) run end to end — detection and mask head both included, nothing given away — on a photo with exactly the kind of touching, same-class instances this lesson has been building toward.

In [ ]:
import torchvision
import cv2
import matplotlib.patches as mpatches

weights = torchvision.models.detection.MaskRCNN_ResNet50_FPN_Weights.COCO_V1
maskrcnn = torchvision.models.detection.maskrcnn_resnet50_fpn(weights=weights)
maskrcnn.eval()  # frozen, no training at all
coco_classes = weights.meta['categories']

photo = cv2.imread('../img/apollo11_crew.jpg')
photo_rgb = cv2.cvtColor(photo, cv2.COLOR_BGR2RGB)

preprocess = weights.transforms()
photo_tensor = torch.tensor(photo_rgb / 255.0, dtype=torch.float32).permute(2, 0, 1)
batch = preprocess(photo_tensor).unsqueeze(0)

with torch.no_grad():
    result = maskrcnn(batch)[0]

score_thresh = 0.8
keep = result['scores'] > score_thresh
labels = result['labels'][keep].numpy()
scores = result['scores'][keep].numpy()
masks = result['masks'][keep, 0].numpy()  # soft per-instance masks

print(f'{keep.sum().item()} instances above {score_thresh:.0%} confidence:')
for lbl, sc in zip(labels, scores):
    print(f'  {coco_classes[lbl]:>10}  {sc:.1%}')

colors = plt.cm.tab10(np.linspace(0, 1, 10))
inst_label_rgb = np.full_like(photo_rgb, 255)
for i in range(len(labels)):
    color = (np.array(colors[i][:3]) * 255).astype(np.uint8)
    inst_label_rgb[masks[i] > 0.5] = color

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
axes[0].imshow(photo_rgb); axes[0].set_title('input'); axes[0].axis('off')
axes[1].imshow(inst_label_rgb); axes[1].set_title('Mask R-CNN per-instance masks'); axes[1].axis('off')
legend_handles = [mpatches.Patch(color=colors[i], label=f'{coco_classes[labels[i]]} #{i+1} ({scores[i]:.0%})')
                  for i in range(len(labels))]
axes[1].legend(handles=legend_handles, loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

<p class="image-attribution">Image source: NASA (public domain), via <a href="https://commons.wikimedia.org/wiki/File:Apollo_11_Crew.jpg" target="_blank" rel="noopener">Wikimedia Commons</a></p>

The three astronauts are shoulder-to-shoulder, touching along multiple arm and torso boundaries, and belong to the exact same COCO class (`person`) — precisely the case where a semantic mask would merge them into one undifferentiated blob. Mask R-CNN separates them cleanly into three distinct instances anyway, because its detector proposes three separate boxes first (one per astronaut) and the mask head only looks inside a single box at a time, exactly like the mask head above. A fourth, spurious detection also shows up: a small `clock` mask at 84% confidence, on the left astronaut's wrist — the round, mirrored cuff-mounted checklist on Armstrong's arm, misread as a clock face.

### Exercises

1. Reduce `dist` in `make_scene` from the range `(6, 9)` to `(2, 5)`, making the circles overlap much more heavily (centers closer together). Does the offset-voting method's instance-count accuracy hold up, or does it start failing too — and if so, what does the failure look like (merged instances, split instances, or something else)?
2. The clustering here uses `peak_dist=4` for non-max suppression on the vote accumulator. Try `peak_dist=8`. Does accuracy improve, get worse, or become sensitive to exactly how close two true circle centers happen to be in a given scene?
3. This lesson only ever has 2 circles per scene. Extend `make_scene` to place a random number of circles (1 to 4) and adjust `cluster_instances`'s loop bound accordingly. Does the offset-voting approach's instance count accuracy hold up as the true number of instances grows?
4. The mask head above was trained and evaluated using the *ground-truth* box center, sidestepping detection entirely. Perturb each crop's center by a few random pixels before cropping (simulating an imperfect detector) and rerun. How much does mask IoU degrade as the box gets less accurately centered — and does that match Mask R-CNN's real-world dependence on detector quality?